# Egypt Property Finder Dataset: Exploratory Data Analysis & Price Insights

## Executive Summary
This notebook explores the **Egypt Property Finder Comprehensive Dataset**. Our goal is to extract meaningful business insights from the real estate market in Egypt, analyzing price distributions, geographical hotspots, and key property features that drive value. 

Finally, we build a baseline Machine Learning model to predict property prices based on available features.

**Use Cases:**
- Real estate investment analysis.
- Predictive pricing models.
- Consumer market trends in Egypt.


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set aesthetic configurations
sns.set_theme(style="whitegrid")


## Section 2 - Data Loading
We will load the primary dataset (`all_egypt.csv`) which contains all the aggregated listings.


In [ ]:
# Define path (Update this to your Kaggle input path)
DATA_PATH = '/kaggle/input/egypt-property-finder-dataset/processed/all_egypt.csv'

try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    # Fallback to local path for testing
    df = pd.read_csv('Egypt_Property_Finder_Kaggle/processed/all_egypt.csv')

print(f"Dataset Shape: {df.shape}")
display(df.head())


## Section 3 - Data Quality Assessment
Let's review missing values, duplicates, and general statistics.


In [ ]:
# Missing Values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Missing Values:\n", missing)

# Duplicates
print("\nDuplicate Rows:", df.duplicated().sum())

# Summary Statistics
display(df.describe())


## Section 4 - Exploratory Data Analysis
Let's visualize the core distributions of our data.


In [ ]:
# Property Type Distribution
prop_counts = df['property_type'].value_counts().reset_index()
prop_counts.columns = ['Property Type', 'Count']
fig = px.bar(prop_counts, x='Property Type', y='Count', title='Distribution of Property Types', color='Count')
fig.show()

# Price Distribution (Log Scale)
fig = px.histogram(df, x='price', nbins=100, title='Property Price Distribution (Log Scale)', log_y=True)
fig.show()

# Bedrooms Distribution
fig = px.histogram(df.dropna(subset=['bedrooms']), x='bedrooms', title='Bedrooms Distribution')
fig.show()

# Size Distribution
fig = px.histogram(df[df['size'] < df['size'].quantile(0.99)], x='size', nbins=50, title='Property Size Distribution (Excluding Top 1% Outliers)')
fig.show()


**Interpretation:**
- Apartments and Villas heavily dominate the Egyptian market.
- The price distribution has a massive long tail, justifying the log scale.
- 3-bedroom properties are by far the most common configuration.


## Section 5 - Geographic Analysis
Where are the most expensive locations?


In [ ]:
# Extract City/Broad location from the detailed location string
df['broad_location'] = df['location'].apply(lambda x: str(x).split(',')[-1].strip() if pd.notnull(x) else 'Unknown')

geo_stats = df.groupby('broad_location').agg({'price': 'median', 'size': 'median', 'id': 'count'}).reset_index()
geo_stats = geo_stats[geo_stats['id'] > 50].sort_values('price', ascending=False) # Filter locations with enough data

fig = px.bar(geo_stats.head(10), x='broad_location', y='price', title='Top 10 Most Expensive Broad Locations (Median Price)')
fig.show()

fig = px.bar(geo_stats.sort_values('size', ascending=False).head(10), x='broad_location', y='size', title='Top 10 Locations by Largest Median Property Size')
fig.show()


## Section 6 - Amenities Analysis
What features are commonly offered?


In [ ]:
# Extract amenities
amenities_series = df['amenities'].dropna().astype(str).str.replace(r'[\[\]\']', '', regex=True).str.split(',')
all_amenities = [item.strip() for sublist in amenities_series for item in sublist if item.strip()]

amenities_counts = pd.Series(all_amenities).value_counts().head(15).reset_index()
amenities_counts.columns = ['Amenity', 'Count']

fig = px.bar(amenities_counts, x='Count', y='Amenity', orientation='h', title='Top 15 Most Common Amenities')
fig.gca().invert_yaxis()
fig.show()


## Section 7 - Correlation Analysis
Understanding feature relationships.


In [ ]:
numeric_cols = ['price', 'bedrooms', 'bathrooms', 'size', 'latitude', 'longitude']
corr = df[numeric_cols].corr()

fig = px.imshow(corr, text_auto=True, aspect="auto", title='Correlation Heatmap')
fig.show()


**Interpretation:**
As expected, `size`, `bedrooms`, and `bathrooms` exhibit high positive correlation with `price`.


## Section 8 - Business Insights
1. **Premium Hubs:** Certain districts dramatically outprice others, heavily skewed by newly developed gated communities (e.g., New Cairo, 6th of October).
2. **Standardization:** The 3-bedroom, 2-bathroom configuration is the undisputed standard for Egyptian families.
3. **Amenities Drive Value:** Properties listing "Security" and "Balcony" are universally present in top-tier listings.


## Section 9 - Baseline Machine Learning
Let's build a fast Random Forest model to establish a price prediction baseline.


In [ ]:
# Prepare data
ml_df = df.dropna(subset=['price', 'bedrooms', 'bathrooms', 'size']).copy()

# Remove extreme outliers for baseline stability
q_low = ml_df['price'].quantile(0.01)
q_hi  = ml_df['price'].quantile(0.99)
ml_df = ml_df[(ml_df['price'] > q_low) & (ml_df['price'] < q_hi)]

features = ['bedrooms', 'bathrooms', 'size']
X = ml_df[features]
y = ml_df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Model
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Predict
preds = rf.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("--- Baseline Random Forest Metrics ---")
print(f"MAE:  {mae:,.2f} EGP")
print(f"RMSE: {rmse:,.2f} EGP")
print(f"R²:   {r2:.4f}")


## Section 10 - Conclusion
We have successfully mapped the Egyptian real estate landscape and established a functional baseline model capable of explaining a significant portion of price variance solely based on physical property dimensions. 

Future improvements could utilize NLP on the property titles and geo-spatial embeddings using latitude/longitude clustering.
